# ReVA Dataset EDA

Loads only the three annotation JSON files from the pinned Hugging Face revision. It does not download videos or expose test answers to a model. The plots describe annotation structure and reference overlap.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from preprocessing import read_annotations
from preprocessing.hf_access import fetch_annotation_file
manifest=json.loads(Path("manifests/reva_manifest.json").read_text(encoding="utf-8"))
repo, revision=manifest["repo_id"], manifest["revision"]
splits={name: read_annotations(fetch_annotation_file(repo, filename, revision))
        for name, filename in [("train","train.json"),("val","val.json"),("test","test.json")]}
print("revision:", revision, "records:", {k:len(v) for k,v in splits.items()})


In [ ]:
frames={name:pd.DataFrame(rows) for name,rows in splits.items()}
summary=[]
for name,df in frames.items():
    paths=df.get("video_path",pd.Series(dtype=str))
    answers=df.get("correct_answer",pd.Series(dtype=str)).fillna("").replace("","<empty>")
    summary.append({"split":name,"questions":len(df),"unique_videos":paths.nunique(),
                    "duplicate_sample_ids":int(df["qa_id"].duplicated().sum()) if "qa_id" in df else None,
                    "missing_video_paths":int(paths.isna().sum()),"unlabeled_answers":int((answers=="<empty>").sum())})
summary_df=pd.DataFrame(summary).set_index("split")
summary_df


In [ ]:
fig, axes=plt.subplots(1,2,figsize=(14,5))
category=pd.concat([df.assign(split=name) for name,df in frames.items()]).groupby(["category","split"]).size().unstack(fill_value=0)
category.plot(kind="bar",ax=axes[0],title="Questions by category and split")
tasks=pd.concat([df.assign(split=name) for name,df in frames.items()]).groupby(["task","split"]).size().unstack(fill_value=0)
tasks.plot(kind="bar",ax=axes[1],title="Questions by task and split")
axes[0].set_ylabel("Questions"); axes[1].set_ylabel("Questions")
plt.tight_layout()


In [ ]:
fig, axes=plt.subplots(1,2,figsize=(12,4))
for name,df in frames.items():
    answers=df["correct_answer"].fillna("").replace("","<empty>").value_counts()
    answers.reindex(["A","B","C","D","<empty>"],fill_value=0).plot(kind="line",marker="o",label=name,ax=axes[0])
    per_video=df.groupby("video_path").size()
    per_video.plot(kind="hist",bins=20,alpha=.45,label=name,ax=axes[1])
axes[0].set_title("Answer label counts"); axes[0].legend()
axes[1].set_title("Questions per video"); axes[1].set_xlabel("Questions")
plt.tight_layout()


In [ ]:
for name,df in frames.items():
    option_errors=df["options"].map(lambda x:not isinstance(x,dict) or set(x)!={"A","B","C","D"}).sum()
    missing_fields=df[["qa_id","video_path","question","options"]].isna().sum().to_dict()
    print(name,{"option_errors":int(option_errors),"missing_core_fields":missing_fields})
train_paths=set(frames["train"]["video_path"]); val_paths=set(frames["val"]["video_path"]); test_paths=set(frames["test"]["video_path"])
print("video overlaps",{"train_val":len(train_paths&val_paths),"train_test":len(train_paths&test_paths),"val_test":len(val_paths&test_paths)})
repo_files=set(manifest["files"])
for name,paths in [(n,set(df["video_path"])) for n,df in frames.items()]:
    missing=paths-repo_files
    print(name,"annotation paths absent from pinned manifest:",len(missing),sorted(missing)[:5])
